In [2]:
# import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
%matplotlib qt 

In [3]:
collar = pd.read_csv('collar.csv')
collar.head()

,Nom,X,Y,Z_g_earth,Longueur
0,PZ1,433400.00,354250.00,458,574.50
1,PZ2,441977.01,358433.96,445,780.00
2,PZ3,446636.77,357501.94,327,551.30
3,PZ4,436850.00,355100.00,466,813.41
4,PZ5,439292.77,356688.05,446,747.55


In [4]:
collar.shape

(132, 5)

In [5]:
collar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132 entries, 0 to 131
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Nom        132 non-null    object 
 1   X          132 non-null    float64
 2   Y          132 non-null    float64
 3   Z_g_earth  132 non-null    int64  
 4   Longueur   132 non-null    float64
dtypes: float64(3), int64(1), object(1)
memory usage: 5.3+ KB


In [6]:
collar.describe()

,X,Y,Z_g_earth,Longueur
count,132.000000,132.000000,132.000000,132.000000
mean,439413.058333,355468.510076,388.590909,645.468333
std,7829.178191,4993.387540,96.892082,190.315077
min,417433.210000,340309.460000,145.000000,167.700000
25%,435600.327500,352592.347500,355.000000,517.325000
50%,438321.125000,356408.025000,441.500000,636.490000
75%,443913.482500,358442.982500,455.000000,777.000000
max,456261.390000,367156.410000,478.000000,1300.000000


In [7]:
collar.dtypes

Nom           object
X            float64
Y            float64
Z_g_earth      int64
Longueur     float64
dtype: object

In [8]:
collar.isna().any()

Nom          False
X            False
Y            False
Z_g_earth    False
Longueur     False
dtype: bool

In [9]:
collar.columns

Index(['Nom', 'X', 'Y', 'Z_g_earth', 'Longueur'], dtype='object')

In [10]:
composited = pd.read_csv('Composites.csv')
composited.head()

,Sample Number,Drillholes,Depth From,Depth To,X,Y,Z,Length,Lithologies
0,1,PZ1,0,1.0,433400.0,354250.0,457.5,1.0,SEDS
1,2,PZ1,1,2.0,433400.0,354250.0,456.5,1.0,SEDS
2,3,PZ1,2,3.0,433400.0,354250.0,455.5,1.0,SEDS
3,4,PZ1,3,4.0,433400.0,354250.0,454.5,1.0,SEDS
4,5,PZ1,4,5.0,433400.0,354250.0,453.5,1.0,SEDS


In [11]:
composited.shape

(85030, 9)

In [12]:
composited.isna().any()

Sample Number    False
Drillholes       False
Depth From       False
Depth To         False
X                False
Y                False
Z                False
Length           False
Lithologies       True
dtype: bool

In [13]:
composited['Lithologies'].isna().sum()

40

In [14]:
composited.dropna(subset=['Lithologies'], how='any', inplace=True)

In [15]:
composited.isna().any()

Sample Number    False
Drillholes       False
Depth From       False
Depth To         False
X                False
Y                False
Z                False
Length           False
Lithologies      False
dtype: bool

In [16]:
print(composited['Lithologies'].value_counts() )

Lithologies
SS      46043
SEDS    16042
BST      9154
DR       9135
SG       3485
SEDI      806
CAL       325
Name: count, dtype: int64


In [17]:
# Get distribution
counts = composited['Lithologies'].value_counts(dropna=False)

# Pie chart
plt.figure(figsize=(10, 6))
counts.plot(kind='pie', autopct='%1.1f%%', startangle=90)
plt.title('Lithology Distribution')
plt.ylabel('')
plt.show()

# Bar chart with percentages
plt.figure(figsize=(12, 6))
percentages = composited['Lithologies'].value_counts(dropna=False, normalize=True) * 100
percentages.plot(kind='bar')
plt.title('Lithology Distribution (%)')
plt.xlabel('Lithology')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=45)
plt.axhline(y=100/len(percentages), color='r', linestyle='--', label='Uniform distribution')
plt.legend()
plt.tight_layout()
plt.show()

In [18]:
counts = composited['Lithologies'].value_counts(dropna=False)
percentages = composited['Lithologies'].value_counts(dropna=False, normalize=True) * 100
cumulative = percentages.cumsum()

distribution = pd.DataFrame({
    'Count': counts,
    'Percentage (%)': round(percentages, 2),
    'Cumulative (%)': round(cumulative, 2)
})
print(distribution)

             Count  Percentage (%)  Cumulative (%)
Lithologies                                       
SS           46043           54.17           54.17
SEDS         16042           18.88           73.05
BST           9154           10.77           83.82
DR            9135           10.75           94.57
SG            3485            4.10           98.67
SEDI           806            0.95           99.62
CAL            325            0.38          100.00


In [19]:
BG  = '#F8F9FA'
PAL = ['#2563EB','#DC2626','#16A34A','#F59E0B','#8B5CF6','#06B6D4','#EC4899']

lith_order = composited['Lithologies'].value_counts().index.tolist()
color_map  = {l: PAL[i % len(PAL)] for i, l in enumerate(lith_order)}

In [20]:
import matplotlib.gridspec as gridspec

In [21]:
# Figure 1 : overview channel 
fig = plt.figure(figsize=(18, 12), facecolor=BG)
fig.suptitle('EDA Overview – Drillhole Database',
             fontsize=18, fontweight='bold', y=1.01)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 1a :ithology bar chart
ax1 = fig.add_subplot(gs[0, 0])
lith = composited['Lithologies'].value_counts()
bars = ax1.barh(lith.index, lith.values,
                color=PAL[:len(lith)], edgecolor='white')
for bar, val in zip(bars, lith.values):
    ax1.text(val + 200, bar.get_y() + bar.get_height() / 2,
             f'{val:,}', va='center', fontsize=9)
ax1.set_xlabel('Number of 1-m intervals', fontsize=9)
ax1.set_title('Lithology Distribution\n(all composites)', fontweight='bold')
ax1.set_facecolor(BG)
for sp in ['top', 'right']: ax1.spines[sp].set_visible(False)

# 1b : drillhole length histogram
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(collar['Longueur'].dropna(), bins=20,
         color='#2563EB', alpha=0.8, edgecolor='white')
ax2.axvline(collar['Longueur'].mean(), color='red', lw=2, ls='--',
            label=f"Mean = {collar['Longueur'].mean():.0f} m")
ax2.set_xlabel('Drillhole Length (m)', fontsize=9)
ax2.set_ylabel('Count', fontsize=9)
ax2.set_title(f"Drillhole Length Distribution\n({len(collar)} drillholes)",
              fontweight='bold')
ax2.legend(fontsize=8)
ax2.set_facecolor(BG)
for sp in ['top', 'right']: ax2.spines[sp].set_visible(False)

# 1c : depth (From) per lithology boxplot
ax3 = fig.add_subplot(gs[0, 2])
data_box = [composited[composited['Lithologies'] == l]['Depth From'].dropna().values
            for l in lith_order]
bp = ax3.boxplot(data_box, vert=True, patch_artist=True,
                 tick_labels=lith_order)
for patch, color in zip(bp['boxes'], PAL):
    patch.set_facecolor(color); patch.set_alpha(0.7)
for med in bp['medians']:
    med.set_color('white'); med.set_linewidth(2)
ax3.set_xlabel('Lithology', fontsize=9)
ax3.set_ylabel('Depth From (m)', fontsize=9)
ax3.set_title('Depth Distribution by Lithology', fontweight='bold')
ax3.set_facecolor(BG)
for sp in ['top', 'right']: ax3.spines[sp].set_visible(False)

# 1d : collar plan view coloured by length
ax4 = fig.add_subplot(gs[1, 0])
sc = ax4.scatter(collar['X'], collar['Y'], c=collar['Longueur'],
                 cmap='YlOrRd', s=50, edgecolors='white', linewidths=0.4)
plt.colorbar(sc, ax=ax4, label='Length (m)', shrink=0.85)
ax4.set_xlabel('Easting (m)',  fontsize=9)
ax4.set_ylabel('Northing (m)', fontsize=9)
ax4.set_title('Drillhole Collar Locations\n(colour = total length)',
              fontweight='bold')
ax4.ticklabel_format(style='sci', axis='both', scilimits=(5, 5))
ax4.set_facecolor(BG)
for sp in ['top', 'right']: ax4.spines[sp].set_visible(False)

# 1e : Z elevation distribution
ax5 = fig.add_subplot(gs[1, 1])
ax5.hist(composited['Z'].dropna(), bins=40,
         color='#16A34A', alpha=0.8, edgecolor='white')
ax5.set_xlabel('Z Elevation (m)', fontsize=9)
ax5.set_ylabel('Count', fontsize=9)
ax5.set_title('Composite Elevation (Z)\nDistribution', fontweight='bold')
ax5.set_facecolor(BG)
for sp in ['top', 'right']: ax5.spines[sp].set_visible(False)

# 1f – Number of composites per drillhole
ax6 = fig.add_subplot(gs[1, 2])
n_intervals = composited.groupby('Drillholes').size()
ax6.hist(n_intervals, bins=25,
         color='#8B5CF6', alpha=0.8, edgecolor='white')
ax6.axvline(n_intervals.mean(), color='red', lw=2, ls='--',
            label=f'Mean = {n_intervals.mean():.0f}')
ax6.set_xlabel('Number of 1-m composites', fontsize=9)
ax6.set_ylabel('Count', fontsize=9)
ax6.set_title('Composites per Drillhole', fontweight='bold')
ax6.legend(fontsize=8)
ax6.set_facecolor(BG)
for sp in ['top', 'right']: ax6.spines[sp].set_visible(False)

plt.savefig(f"EDA_1_overview.png", dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 1 saved – Overview dashboard")

✓ Figure 1 saved – Overview dashboard


In [22]:
# figure 2 : spatial distribution of lithologies / by colors
fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=BG)
fig.suptitle('Spatial Distribution by Lithology',
             fontsize=15, fontweight='bold')

for lith in lith_order:
    sub = composited[composited['Lithologies'] == lith]
    axes[0].scatter(sub['X'], sub['Y'],
                    label=lith, s=2, alpha=0.4, color=color_map[lith])
    axes[1].scatter(sub['X'], sub['Z'],
                    label=lith, s=2, alpha=0.4, color=color_map[lith])

for ax, (xl, yl, title) in zip(axes, [
        ('Easting (m)', 'Northing (m)', 'Plan View  (X vs Y)'),
        ('Easting (m)', 'Elevation Z (m)', 'Section View  (X vs Z)')]):
    ax.set_xlabel(xl, fontsize=10)
    ax.set_ylabel(yl, fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.ticklabel_format(style='sci', axis='x', scilimits=(5, 5))
    ax.set_facecolor(BG)
    for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
axes[0].legend(markerscale=5, fontsize=9, title='Lithology')
plt.tight_layout()
plt.savefig(f"EDA_2_spatial.png", dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 2 saved – Spatial distribution")

✓ Figure 2 saved – Spatial distribution


In [23]:
#figure 3 : correlation heatmap
num_df = composited[['Depth From', 'Depth To', 'X', 'Y', 'Z', 'Length']].dropna()
corr   = num_df.corr()
mask   = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 11}, ax=ax,
            cbar_kws={'shrink': 0.8, 'label': 'Pearson r'})
ax.set_title('Correlation Matrix – Composite Numeric Variables',
             fontsize=13, fontweight='bold', pad=12)
ax.set_facecolor(BG)
plt.tight_layout()
plt.savefig(f"EDA_3_correlation.png", dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 3 saved – Correlation heatmap")

✓ Figure 3 saved – Correlation heatmap


In [24]:
# another correlation plot with all columns
# encode text columns so they can be correlated
composited['Lithology_code'] = pd.Categorical(composited['Lithologies']).codes
composited['Drillhole_code'] = pd.Categorical(composited['Drillholes']).codes
 
cols = ['Sample Number', 'Depth From', 'Depth To', 'X', 'Y', 'Z',
        'Length', 'Lithology_code', 'Drillhole_code']
 
corr = composited[cols].corr()
 
# cleaner axis labels
rename = {
    'Sample Number' : 'Sample №',
    'Depth From'    : 'Depth From',
    'Depth To'      : 'Depth To',
    'X'             : 'X',
    'Y'             : 'Y',
    'Z'             : 'Z',
    'Length'        : 'Length',
    'Lithology_code': 'Lithology',
    'Drillhole_code': 'Drillhole',
}
corr.index   = [rename[c] for c in corr.index]
corr.columns = [rename[c] for c in corr.columns]
 
BG = '#F8F9FA'
fig, ax = plt.subplots(figsize=(11, 9), facecolor=BG)
ax.set_facecolor(BG)
 
mask = np.triu(np.ones_like(corr, dtype=bool))
 
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.8, linecolor='white',
            annot_kws={'size': 11, 'weight': 'bold'}, ax=ax,
            cbar_kws={'shrink': 0.75, 'label': 'Pearson r'})
 
ax.set_title('Correlation Matrix – All Columns\n(Lithologies and Drillholes are label-encoded)',
             fontsize=14, fontweight='bold', pad=14)
ax.tick_params(axis='x', rotation=45, labelsize=11)
ax.tick_params(axis='y', rotation=0,  labelsize=11)
 
plt.tight_layout()
plt.savefig('EDA_correlation_all.png', dpi=160, bbox_inches='tight')
plt.close()

In [25]:
# 3d example of the drillhole and the where the sg layer mostly spreads 
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D

comp = composited.dropna(subset=['X', 'Y', 'Z', 'Lithologies']).reset_index(drop=True)
comp['label'] = (comp['Lithologies'] == 'SG').astype(int)
 
fig = plt.figure(figsize=(14, 10), facecolor='#0F172A')
ax  = fig.add_subplot(111, projection='3d')
ax.set_facecolor('#0F172A')
fig.patch.set_facecolor('#0F172A')
 
for bh, grp in comp.groupby('Drillholes'):
    grp = grp.sort_values('Depth From')
 
    # drillhole trace
    ax.plot(grp['X'], grp['Y'], grp['Z'],
            color='#334155', lw=0.5, alpha=0.6)
 
    # non-SG points
    non_sg = grp[grp['label'] == 0]
    ax.scatter(non_sg['X'], non_sg['Y'], non_sg['Z'],
               c='#64748B', s=1, alpha=0.3)
 
    # SG points
    sg = grp[grp['label'] == 1]
    if len(sg) > 0:
        ax.scatter(sg['X'], sg['Y'], sg['Z'],
                   c='#F59E0B', s=8, alpha=1.0, zorder=5)
 
ax.set_xlabel('X (m)', color='#CBD5E1', fontsize=9, labelpad=8)
ax.set_ylabel('Y (m)', color='#CBD5E1', fontsize=9, labelpad=8)
ax.set_zlabel('Z (m)', color='#CBD5E1', fontsize=9, labelpad=8)
ax.tick_params(colors='#64748B', labelsize=7)
ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False
ax.xaxis.pane.set_edgecolor('#1E293B')
ax.yaxis.pane.set_edgecolor('#1E293B')
ax.zaxis.pane.set_edgecolor('#1E293B')
ax.grid(True, color='#1E293B', linewidth=0.5)
 
ax.set_title('3D Drillhole Distribution – SG Layer\n'
             'Yellow = SG (label 1)   Grey = Non-SG (label 0)',
             fontsize=13, fontweight='bold', color='white', pad=15)
 
legend = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#F59E0B',
           markersize=10, label='SG  (label = 1)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#64748B',
           markersize=10, label='Non-SG  (label = 0)'),
]
ax.legend(handles=legend, facecolor='#1E293B', labelcolor='white', fontsize=10)
 
plt.tight_layout()
plt.savefig('drillholes_3d.png', dpi=160, bbox_inches='tight', facecolor='#0F172A')
plt.close()

In [26]:
# now lets try to plot the graph and the connections

from sklearn.neighbors import NearestNeighbors
K      = 16
coords = collar[['X', 'Y']].values
 
nbrs = NearestNeighbors(n_neighbors=K + 1, algorithm='ball_tree').fit(coords)
_, indices = nbrs.kneighbors(coords)
indices = indices[:, 1:]   # drop self
 
fig, ax = plt.subplots(figsize=(10, 9))
 
# edges
for i in range(len(collar)):
    x0, y0 = collar.loc[i, 'X'], collar.loc[i, 'Y']
    for j in indices[i]:
        x1, y1 = collar.loc[j, 'X'], collar.loc[j, 'Y']
        ax.plot([x0, x1], [y0, y1],
                color='black', lw=0.6, alpha=0.5, zorder=1)
 
# nodes coloured by Z elevation
sc = ax.scatter(collar['X'], collar['Y'], c=collar['Z_g_earth'],
                cmap='viridis', s=40, zorder=2,
                edgecolors='black', linewidths=0.4)
 
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Z Depth', fontsize=10)
 
ax.set_xlabel('XCOLLAR', fontsize=11)
ax.set_ylabel('YCOLLAR', fontsize=11)
ax.set_title(f'Two-dimensional graph visualisation of the sampled drillhole connections  (k={K})',
             fontsize=11)
ax.ticklabel_format(style='sci', axis='both', scilimits=(5, 5))
 
plt.tight_layout()
plt.savefig('graph_2d_collars.png', dpi=160, bbox_inches='tight')
plt.close()

## SS Layer – Hanging Wall & Foot Wall Extraction

All drillholes intersect the **SS** lithology. For each drillhole we extract:
- **Hanging Wall (HW)** = highest Z elevation (top / roof of the SS layer)
- **Foot Wall  (FW)**   = lowest  Z elevation (bottom / floor of the SS layer)
- **Thickness**         = HW − FW

The **unsampled targets** are not other drillholes — they are arbitrary **(X, Y) locations within the study area that were never drilled**. The goal is to interpolate HW and FW surfaces across a dense spatial grid, then compute a full **3D model of the SS layer**.

In [27]:
import networkx as nx
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error
from matplotlib.patches import Patch

# ── Extract SS HW and FW per drillhole ──────────────────────────────────────
ss = composited[composited['Lithologies'] == 'SS'].copy()

ss_bounds = ss.groupby('Drillholes').agg(
    HW        = ('Z', 'max'),                          # top  of SS (highest elevation)
    FW        = ('Z', 'min'),                          # base of SS (lowest  elevation)
    Thickness = ('Z', lambda x: x.max() - x.min()),
    X         = ('X', 'first'),
    Y         = ('Y', 'first'),
).reset_index()

print(f"Drillholes with SS data : {len(ss_bounds)} / {len(collar)}")
print()
print(ss_bounds[['HW', 'FW', 'Thickness']].describe().round(1))

# ── Spatial overview of HW, FW, and Thickness ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor=BG)
fig.suptitle('SS Layer at Drillhole Locations – HW, FW, and Thickness',
             fontsize=13, fontweight='bold')

for ax, col, title, cmap in zip(
        axes,
        ['HW', 'FW', 'Thickness'],
        ['Hanging Wall Z (m)', 'Foot Wall Z (m)', 'Thickness (m)'],
        ['Blues_r', 'Reds_r', 'YlOrRd']):
    sc = ax.scatter(ss_bounds['X'], ss_bounds['Y'], c=ss_bounds[col],
                    cmap=cmap, s=60, edgecolors='k', linewidths=0.4, zorder=3)
    plt.colorbar(sc, ax=ax, label=title, shrink=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Easting (m)',  fontsize=9)
    ax.set_ylabel('Northing (m)', fontsize=9)
    ax.ticklabel_format(style='sci', axis='both', scilimits=(5, 5))
    ax.set_facecolor(BG)
    for sp in ['top', 'right']: ax.spines[sp].set_visible(False)

plt.tight_layout()
plt.savefig('ss_hw_fw_spatial.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n✓ ss_hw_fw_spatial.png saved")

Drillholes with SS data : 122 / 132

          HW     FW  Thickness
count  122.0  122.0      122.0
mean   201.7 -174.7      376.4
std     93.3  184.8      169.0
min    -53.5 -707.5       37.0
25%    146.0 -267.8      215.5
50%    210.0 -178.0      387.0
75%    262.2  -16.0      505.0
max    387.5  237.5      723.0

✓ ss_hw_fw_spatial.png saved


In [28]:
# ── K-Selection Sweep ────────────────────────────────────────────────────────
# For each K we measure:
#   1. Graph connectivity (how many disconnected components remain)
#   2. Leave-One-Out CV error using Inverse Distance Weighting (IDW)
#      — this simulates how well a K-neighbour graph predicts held-out nodes
#   3. Mean and 95th-pct edge length (spatial footprint of connections)

coords   = ss_bounds[['X', 'Y']].values
hw_vals  = ss_bounds['HW'].values
fw_vals  = ss_bounds['FW'].values
n_nodes  = len(coords)

K_range = range(2, min(26, n_nodes))
records = []

for K in K_range:
    nbrs = NearestNeighbors(n_neighbors=K + 1, algorithm='ball_tree').fit(coords)
    dists, idxs = nbrs.kneighbors(coords)
    dists = dists[:, 1:]    # exclude self
    idxs  = idxs[:,  1:]

    # build graph to check connectivity
    G = nx.Graph()
    G.add_nodes_from(range(n_nodes))
    for i in range(n_nodes):
        for j_pos, j in enumerate(idxs[i]):
            G.add_edge(i, int(j), weight=float(dists[i, j_pos]))
    n_comp    = nx.number_connected_components(G)
    connected = (n_comp == 1)

    # LOO-CV: predict each node from its K neighbours using IDW
    hw_pred, fw_pred = [], []
    for i in range(n_nodes):
        w = 1.0 / (dists[i] + 1e-6)
        w /= w.sum()
        hw_pred.append(np.dot(w, hw_vals[idxs[i]]))
        fw_pred.append(np.dot(w, fw_vals[idxs[i]]))

    records.append({
        'K'         : K,
        'n_comp'    : n_comp,
        'connected' : connected,
        'HW_MAE'    : mean_absolute_error(hw_vals, hw_pred),
        'FW_MAE'    : mean_absolute_error(fw_vals, fw_pred),
        'HW_RMSE'   : np.sqrt(np.mean((hw_vals - np.array(hw_pred))**2)),
        'FW_RMSE'   : np.sqrt(np.mean((fw_vals - np.array(fw_pred))**2)),
        'mean_dist' : dists.mean(),
        'p95_dist'  : np.percentile(dists, 95),
    })

res = pd.DataFrame(records)

k_connected = int(res.loc[res['connected'], 'K'].min())
k_best_hw   = int(res.loc[res['HW_MAE'].idxmin(), 'K'])
k_best_fw   = int(res.loc[res['FW_MAE'].idxmin(), 'K'])
# recommended K: at least fully connected, and near the MAE elbow (cap at 15)
k_rec = max(k_connected, min(k_best_hw, k_best_fw, 15))

print(f"Min K for full connectivity : {k_connected}")
print(f"K minimising HW LOO-MAE    : {k_best_hw}")
print(f"K minimising FW LOO-MAE    : {k_best_fw}")
print(f"Recommended K              : {k_rec}")
print()
print(res[['K','n_comp','connected','HW_MAE','FW_MAE','mean_dist']].to_string(index=False))

Min K for full connectivity : 3
K minimising HW LOO-MAE    : 7
K minimising FW LOO-MAE    : 6
Recommended K              : 6

 K  n_comp  connected    HW_MAE    FW_MAE   mean_dist
 2       4      False 44.563791 68.283567 1364.507702
 3       1       True 42.786325 62.182253 1526.061447
 4       1       True 42.068425 60.596766 1664.260698
 5       1       True 42.498133 59.371164 1795.578085
 6       1       True 42.828793 58.242570 1916.985062
 7       1       True 42.040432 59.119120 2031.715823
 8       1       True 42.604823 59.504179 2141.414101
 9       1       True 42.571946 60.640707 2247.560261
10       1       True 42.736435 61.017301 2348.989165
11       1       True 43.060662 62.415960 2447.574452
12       1       True 43.020474 63.405893 2540.827499
13       1       True 43.568808 64.285530 2631.515744
14       1       True 43.773541 65.382766 2722.031677
15       1       True 44.027348 66.430142 2810.023412
16       1       True 44.120823 67.054763 2897.472172
17       1

In [29]:
# ── K-Selection Dashboard ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor=BG)
fig.suptitle('K-Selection Analysis – SS Hanging Wall & Foot Wall Graph',
             fontsize=14, fontweight='bold')

for ax in axes.flat:
    ax.set_facecolor(BG)
    for sp in ['top', 'right']: ax.spines[sp].set_visible(False)

# shared vertical lines
vline_kw  = dict(color='#16A34A', ls='--', lw=1.8, label=f'Min connected (K={k_connected})')
vrec_kw   = dict(color='#F59E0B', ls=':',  lw=2.0, label=f'Recommended  (K={k_rec})')

# Panel A – LOO MAE
ax = axes[0, 0]
ax.plot(res['K'], res['HW_MAE'], 'o-', color='#2563EB', lw=2, label='HW MAE')
ax.plot(res['K'], res['FW_MAE'], 's-', color='#DC2626', lw=2, label='FW MAE')
ax.axvline(k_connected, **vline_kw)
ax.axvline(k_rec,       **vrec_kw)
ax.set_xlabel('K (neighbours)', fontsize=10)
ax.set_ylabel('MAE  (m)', fontsize=10)
ax.set_title('LOO-CV Prediction Error (IDW)', fontweight='bold')
ax.legend(fontsize=9)

# Panel B – LOO RMSE
ax = axes[0, 1]
ax.plot(res['K'], res['HW_RMSE'], 'o-', color='#2563EB', lw=2, label='HW RMSE')
ax.plot(res['K'], res['FW_RMSE'], 's-', color='#DC2626', lw=2, label='FW RMSE')
ax.axvline(k_connected, **vline_kw)
ax.axvline(k_rec,       **vrec_kw)
ax.set_xlabel('K (neighbours)', fontsize=10)
ax.set_ylabel('RMSE  (m)', fontsize=10)
ax.set_title('LOO-CV RMSE', fontweight='bold')
ax.legend(fontsize=9)

# Panel C – graph connectivity
ax = axes[1, 0]
colors = ['#16A34A' if c else '#DC2626' for c in res['connected']]
ax.bar(res['K'], res['n_comp'], color=colors, edgecolor='white')
ax.axhline(1, color='black', lw=1, ls='--')
ax.set_xlabel('K (neighbours)', fontsize=10)
ax.set_ylabel('Connected components', fontsize=10)
ax.set_title('Graph Connectivity  (green = fully connected)', fontweight='bold')
ax.legend(handles=[Patch(color='#16A34A', label='Connected'),
                   Patch(color='#DC2626', label='Fragmented')], fontsize=9)

# Panel D – edge length footprint
ax = axes[1, 1]
ax.plot(res['K'], res['mean_dist'] / 1000, 'o-', color='#8B5CF6',
        lw=2, label='Mean edge length')
ax.plot(res['K'], res['p95_dist']  / 1000, 's--', color='#F59E0B',
        lw=2, label='P95 edge length')
ax.axvline(k_connected, **vline_kw)
ax.axvline(k_rec,       **vrec_kw)
ax.set_xlabel('K (neighbours)', fontsize=10)
ax.set_ylabel('Distance  (km)', fontsize=10)
ax.set_title('Edge Length vs K  (spatial reach)', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('k_selection_ss_hwfw.png', dpi=160, bbox_inches='tight')
plt.close()
print("✓ k_selection_ss_hwfw.png saved")

# Summary
hw_at_rec = res.loc[res['K'] == k_rec, 'HW_MAE'].values[0]
fw_at_rec = res.loc[res['K'] == k_rec, 'FW_MAE'].values[0]
dist_at_rec = res.loc[res['K'] == k_rec, 'mean_dist'].values[0]
print(f"\n── Summary ─────────────────────────────────────────────────────────────")
print(f"  Min K for full connectivity : {k_connected}")
print(f"  K minimising HW LOO-MAE    : {k_best_hw}")
print(f"  K minimising FW LOO-MAE    : {k_best_fw}")
print(f"  ➜ Recommended K            : {k_rec}")
print(f"    HW LOO-MAE  @ K={k_rec}      : {hw_at_rec:.1f} m")
print(f"    FW LOO-MAE  @ K={k_rec}      : {fw_at_rec:.1f} m")
print(f"    Mean edge   @ K={k_rec}      : {dist_at_rec/1000:.2f} km")

✓ k_selection_ss_hwfw.png saved

── Summary ─────────────────────────────────────────────────────────────
  Min K for full connectivity : 3
  K minimising HW LOO-MAE    : 7
  K minimising FW LOO-MAE    : 6
  ➜ Recommended K            : 6
    HW LOO-MAE  @ K=6      : 42.8 m
    FW LOO-MAE  @ K=6      : 58.2 m
    Mean edge   @ K=6      : 1.92 km


In [30]:
# ── Final graph at recommended K + dense prediction grid ─────────────────────
# The grey dots show the undrilled (X, Y) locations we will eventually predict.
# They form a regular grid covering the study area bounding box.

K_final = k_rec

nbrs_final = NearestNeighbors(n_neighbors=K_final + 1, algorithm='ball_tree').fit(coords)
_, idxs_final = nbrs_final.kneighbors(coords)
idxs_final = idxs_final[:, 1:]

# build a regular prediction grid over the study area
margin = 500   # metres padding around the drillhole cloud
x_min, x_max = ss_bounds['X'].min() - margin, ss_bounds['X'].max() + margin
y_min, y_max = ss_bounds['Y'].min() - margin, ss_bounds['Y'].max() + margin
grid_step = 500   # metres between grid nodes
gx = np.arange(x_min, x_max, grid_step)
gy = np.arange(y_min, y_max, grid_step)
GX, GY = np.meshgrid(gx, gy)
grid_pts = np.column_stack([GX.ravel(), GY.ravel()])
print(f"Prediction grid : {len(grid_pts):,} points  ({grid_step} m spacing)")

fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=BG)
fig.suptitle(f'SS Layer Graph  (K={K_final})  –  drillhole nodes + undrilled prediction grid',
             fontsize=13, fontweight='bold')

for ax, col, title, cmap in zip(
        axes,
        ['HW', 'FW'],
        [f'Hanging Wall Z (m)  |  K={K_final}', f'Foot Wall Z (m)  |  K={K_final}'],
        ['Blues_r', 'Reds_r']):

    # prediction grid (undrilled locations)
    ax.scatter(grid_pts[:, 0], grid_pts[:, 1],
               c='#CBD5E1', s=4, alpha=0.4, zorder=1, label='Undrilled grid')

    # KNN edges
    for i in range(len(ss_bounds)):
        x0, y0 = ss_bounds.iloc[i]['X'], ss_bounds.iloc[i]['Y']
        for j in idxs_final[i]:
            x1, y1 = ss_bounds.iloc[j]['X'], ss_bounds.iloc[j]['Y']
            ax.plot([x0, x1], [y0, y1], color='#475569', lw=0.5, alpha=0.5, zorder=2)

    # drillhole nodes coloured by HW or FW
    sc = ax.scatter(ss_bounds['X'], ss_bounds['Y'], c=ss_bounds[col],
                    cmap=cmap, s=60, edgecolors='k', linewidths=0.4, zorder=3,
                    label=f'Drillholes  (n={len(ss_bounds)})')

    plt.colorbar(sc, ax=ax, label='Elevation Z (m)', shrink=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Easting (m)',  fontsize=9)
    ax.set_ylabel('Northing (m)', fontsize=9)
    ax.ticklabel_format(style='sci', axis='both', scilimits=(5, 5))
    ax.set_facecolor(BG)
    ax.legend(fontsize=8, loc='lower right')
    for sp in ['top', 'right']: ax.spines[sp].set_visible(False)

plt.tight_layout()
plt.savefig(f'ss_graph_K{K_final}_with_grid.png', dpi=160, bbox_inches='tight')
plt.close()
print(f"✓ ss_graph_K{K_final}_with_grid.png saved")

Prediction grid : 3,927 points  (500 m spacing)
✓ ss_graph_K6_with_grid.png saved


In [ ]:
def find_outliers(group):
    q1 = group['Depth From'].quantile(0.25)
    q3 = group['Depth From'].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return group[(group['Depth From'] < lower_bound) | (group['Depth From'] > upper_bound)]

outliers = composited.groupby('Lithologies').apply(find_outliers)
print(outliers[['Drillholes','Lithologies']])